## Part 3 — How this dataset became a trained policy

We fine-tuned **SmolVLA** — a small (~500M parameter) vision-language-action model that LeRobot publishes a pretrained base checkpoint for (`lerobot/smolvla_base`) — on the dataset from Part 2.

The idea in one sentence: SmolVLA already knows *roughly* how to look at a camera image and a language instruction and produce robot joint targets, from having been pretrained on many robots and tasks; fine-tuning specializes it to *our* dataset, *our* camera, *our* arm.

The actual command we ran (informational — you won't run this today, there's no GPU at your seat):

```bash
lerobot-train \
  --dataset.repo_id=sohrabark/abc_mhp_v2_merged_20260805 \
  --policy.path=lerobot/smolvla_base \
  --policy.device=cuda \
  --steps=20000 \
  --batch_size=64 \
  --save_freq=5000 \
  --rename_map='{"observation.images.GrispCamera":"observation.images.camera1"}' \
  --wandb.enable=false \
  --policy.push_to_hub=true \
  --policy.repo_id=sohrabark/smolvla_abc_mhp_v2_merged_20260805
```

What each piece roughly means:
- `--dataset.repo_id` — the dataset from Part 2.
- `--policy.path=lerobot/smolvla_base` — start from the pretrained checkpoint, don't train from scratch.
- `--steps` / `--batch_size` — how many gradient updates, and how many examples per update. 20,000 steps × batch 64 over ~25k frames works out to roughly 51 passes over the dataset.
- `--rename_map` — our camera is called `GrispCamera`; SmolVLA expects a generically-named `camera1`, so we rename it at training time.
- `--policy.push_to_hub` — when training finishes, upload the result to the Hub automatically.

What actually happened when we ran it: **~96 minutes** on a single cloud GPU (an NVIDIA L4, 24GB), training loss dropped smoothly from ~0.7 down to **~0.031**.

The result is public on the Hub:

**`sohrabark/smolvla_abc_mhp_v2_merged_20260805`**

You'll use that exact checkpoint in the next part — running on a GPU we've already set up, not yours.

## Part 4 — Run the trained policy on your robot

Here's the piece that makes this workable without a GPU at every seat: the trained policy runs on **one shared cloud GPU**, and your laptop runs a *thin client* — it streams your camera image and your arm's current joint positions over the network, and gets back where to move next, live, at 30 times per second.

```mermaid
flowchart LR
    subgraph laptop["Your laptop"]
        robot["SO-101 arm + camera"]
        client["robot_client"]
        robot <--> client
    end

    subgraph cloud["Shared GPU (cloud)"]
        server["SmolVLA policy server<br/>(smolvla_abc_mhp_v2_merged_20260805)"]
    end

    client -->|"observations<br/>(image + joint positions)<br/>via SSH tunnel"| server
    server -->|"actions @ 30Hz"| client
```

### Two robot arms, one GPU

We have two LeRobot arm sets and one GPU box for the whole room. One GPU can run the policy for both arms at once — we just gave each table **its own port** on the same machine, so the two tables can't interfere with each other:

| Table | Port |
|---|---|
| Table A | `8080` |
| Table B | `8081` |

⚠️ Use **only** the one port matching your table — never the other one, and never anything besides these two. Ask an organizer which table you're on if you're not sure.

### Before you connect

- **Check `lerobot` is installed with the async extras** in the Python environment you'll run `robot_client` from (this can be a different environment from Part 1's LeLab install). If you haven't installed it yet, or Step 2 below fails with a missing-dependency error, run:
  ```bash
  python -m pip install 'lerobot[async]'
  ```
- **Have ready the `robot_port`, calibration `id`, and `camera_index`** you wrote down during calibration in Part 1 — Step 2 below asks for these exact values.

### Connect

**Step 1 — open a tunnel to the shared GPU:**

Your organizer will hand you the `SERVER_IP` and a private key file named `workshop_key` — it only allows port-forwarding, nothing else. Save it locally and restrict its permissions (SSH requires this):

```bash
chmod 600 workshop_key
```

Then open the tunnel, run in its own terminal, pointing `-i` at that key file:

```bash
ssh -i workshop_key -N -L <PORT>:localhost:<PORT> ubuntu@<SERVER_IP>
```

Leave this running for the whole workshop — it's your private pipe into the shared GPU. `<PORT>` is the single port your table was assigned (see the table above — pick one, not both).

`<SERVER_IP>` will be given out at the start of the workshop, alongside `workshop_key`.

**Step 2 — run one task:**

```bash
python -m lerobot.async_inference.robot_client \
  --robot.type=so101_follower \
  --robot.port=<YOUR_ROBOT_PORT> \
  --robot.id=<YOUR_CALIBRATION_ID> \
  --robot.cameras="{ camera1: {type: opencv, index_or_path: <YOUR_CAMERA_INDEX>, width: 640, height: 480, fps: 30}}" \
  --server_address=127.0.0.1:<PORT> \
  --policy_type=smolvla \
  --pretrained_name_or_path=sohrabark/smolvla_abc_mhp_v2_merged_20260805 \
  --policy_device=cuda \
  --client_device=cpu \
  --actions_per_chunk=50 \
  --chunk_size_threshold=0.5 \
  --aggregate_fn_name=weighted_average \
  --fps=30 \
  --task="Pick up the M letter and place it in the first box."
```

Notes:
- `--robot.port`, `--robot.id`, `--robot.cameras` are the `robot_port` / calibration `id` / `camera_index` values you wrote down in Part 1 — LeLab printed them during calibration.
- `--server_address` points at your **local** end of the SSH tunnel — `127.0.0.1`, not `<SERVER_IP>` — the tunnel does the forwarding. `<PORT>` here must match the same single port you used in Step 1.
- This command **runs until you press Ctrl-C** — it has no built-in stopping point. Let it run for ~20-25 seconds (that's roughly how long the training recordings were), then stop it.
- Swap `--task` for any of the three exact strings from the table above to try a different letter.

### The three tasks this policy knows

| Letter | Task string (exact) | Goes into |
|---|---|---|
| **M** | `Pick up the M letter and place it in the first box.` | 1st box |
| **H** | `Pick up the H letter and place it in the second box.` | 2nd box |
| **P** | `Pick up the P letter and place it in the third box.` | 3rd box |

Only these three letters were trained. Anything else is not something the policy has ever seen.

Try running one. Watch what the arm actually does versus what you expected — that observation is useful for the challenge ahead.

## What's next

Open **`workshop_2_challenge.ipynb`**. You have everything you need already: the tunnel command and the single-task inference command above. The challenge is what you build on top of them.